## Modelo de predicción del consumo eléctrico en viviendas(un solo hogar) con datos de ENCEVI

### Objetivo:

Crear un modelo que permita predicir el consumo de energía por vivienda (un hogar) considerando caracteristicas de la vivienda como: existencia de refrigerador, número de focos, número de tvs, número de habitantes entre otras. 

### conclusión 

Dentro del modelo la variable que más influye en el consumo eléctrico por vivienda es la existencia de aire acondicionado, suena lógico y poco influye si se tiene o no refrigerador.

El modelo parece no ser tan útil, pues presenta un error relativo de 67%. Se esplorará que otras caracteristicas pueden ser añadidas para entrenar al modelo y llegar a un eror relativo de 20%-30%.

In [1407]:
#librerías

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error


In [ ]:
#Leer archivos

# información acerca de uso de aire acondicionado por hogar
aa = pd.read_csv("aireacond.csv")
# información número de personas por hogar
cp = pd.read_csv("persona.csv")
# información sobre uso de calefactor en hogares
c = pd.read_csv("calefactor.csv")
# información de número de focos por hogar
f = pd.read_csv("focos.csv")

e= pd.read_csv("encevi.csv", na_values=[" "])
#información vivienda: región
v=pd.read_csv("vivienda.csv")
 

C:\Users\pc\AppData\Local\Temp\ipykernel_18760\525130413.py:12: DtypeWarning: Columns (0: ais_otro_e, 1: ta_otro, 2: otro_c, 3: otro_s, 4: otra_estuf, 5: otro_comb, 6: otra_panel) have mixed types. Specify dtype option on import or set low_memory=False.
  e= pd.read_csv("D:/Georgina RC/inegi/energía/3_encevi_2018_base_de_datos_csv/encevi.csv", na_values=[" "])


### 1.- DF aire acondicionado

In [1409]:
aa.info()

<class 'pandas.DataFrame'>
RangeIndex: 7567 entries, 0 to 7566
Data columns (total 23 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   folio       7567 non-null   int64
 1   foliohog    7567 non-null   int64
 2   id_aire     7567 non-null   int64
 3   aire_tipo   7567 non-null   int64
 4   aire_otro   7567 non-null   str  
 5   aire_capa   7567 non-null   int64
 6   aire_dia    7567 non-null   int64
 7   aire_ene    7567 non-null   str  
 8   aire_feb    7567 non-null   str  
 9   aire_mar    7567 non-null   str  
 10  aire_abr    7567 non-null   str  
 11  aire_may    7567 non-null   str  
 12  aire_jun    7567 non-null   str  
 13  aire_jul    7567 non-null   str  
 14  aire_ago    7567 non-null   str  
 15  aire_sep    7567 non-null   str  
 16  aire_oct    7567 non-null   str  
 17  aire_nov    7567 non-null   str  
 18  aire_dic    7567 non-null   str  
 19  aire_hor    7567 non-null   int64
 20  aire_min    7567 non-null   int64
 21  ai

In [1410]:
aa.isnull().sum()

folio         0
foliohog      0
id_aire       0
aire_tipo     0
aire_otro     0
aire_capa     0
aire_dia      0
aire_ene      0
aire_feb      0
aire_mar      0
aire_abr      0
aire_may      0
aire_jun      0
aire_jul      0
aire_ago      0
aire_sep      0
aire_oct      0
aire_nov      0
aire_dic      0
aire_hor      0
aire_min      0
aire_adqui    0
aire_anio     0
dtype: int64

In [1411]:
aa.duplicated().sum()

np.int64(0)

In [1412]:
# filtrar información útil del DF de aire acondicionado

aire_a=aa[["folio","id_aire"]]
aire_a.sample(3)

,folio,id_aire
1320,3024,2
5231,22753,3
7066,25039,1


In [1413]:
print(aire_a)

      folio  id_aire
0        69        1
1        69        2
2        70        2
3        70        1
4        76        1
...     ...      ...
7562  28010        1
7563  28340        1
7564  28357        1
7565  28626        1
7566  28934        1

[7567 rows x 2 columns]


In [1414]:
aire_a["folio"].duplicated().sum()

np.int64(2342)

In [1415]:
aire_acond=aire_a.groupby("folio")["id_aire"].max().reset_index()

print(aire_acond)

      folio  id_aire
0        69        2
1        70        2
2        76        1
3        84        1
4        97        1
...     ...      ...
5220  28010        1
5221  28340        1
5222  28357        1
5223  28626        1
5224  28934        1

[5225 rows x 2 columns]


In [1416]:
aire_acond["folio"].duplicated().sum()

np.int64(0)

### 2.- DF con información de número de personas en la vivienda

In [1417]:
cp.info()

<class 'pandas.DataFrame'>
RangeIndex: 107239 entries, 0 to 107238
Data columns (total 16 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   folio       107239 non-null  int64
 1   foliohog    107239 non-null  int64
 2   id_pobla    107239 non-null  int64
 3   edad        107239 non-null  int64
 4   nacio_dia   107239 non-null  int64
 5   nacio_mes   107239 non-null  int64
 6   sexo        107239 non-null  int64
 7   parentesco  107239 non-null  int64
 8   asiste_esc  107239 non-null  str  
 9   tipo_esc    107239 non-null  str  
 10  turno_esc   107239 non-null  str  
 11  no_asiste   107239 non-null  str  
 12  grado_inst  107239 non-null  str  
 13  nivel_inst  107239 non-null  str  
 14  alfabetism  107239 non-null  str  
 15  edo_conyug  107239 non-null  str  
dtypes: int64(8), str(8)
memory usage: 13.1 MB


In [1418]:
cp.isnull().sum()

folio         0
foliohog      0
id_pobla      0
edad          0
nacio_dia     0
nacio_mes     0
sexo          0
parentesco    0
asiste_esc    0
tipo_esc      0
turno_esc     0
no_asiste     0
grado_inst    0
nivel_inst    0
alfabetism    0
edo_conyug    0
dtype: int64

In [1419]:
cp.duplicated().sum()

np.int64(0)

In [1420]:
personas=cp[["folio","id_pobla"]]
print(personas)


        folio  id_pobla
0           1         1
1           2         1
2           2         2
3           2         3
4           2         4
...       ...       ...
107234  28953         2
107235  28953         3
107236  28953         4
107237  28953         5
107238  28953         6

[107239 rows x 2 columns]


In [1421]:
personas["folio"].duplicated().sum()

np.int64(78286)

In [1422]:
habitantes=personas.groupby("folio")["id_pobla"].max().reset_index()
print(habitantes)

       folio  id_pobla
0          1         1
1          2         4
2          3         4
3          4         1
4          5         2
...      ...       ...
28948  28949         4
28949  28950         3
28950  28951         2
28951  28952         4
28952  28953         6

[28953 rows x 2 columns]


In [1423]:
habitantes["folio"].duplicated().sum()

np.int64(0)

### 3.- DF con información de calefactor

In [1424]:
c.info()


<class 'pandas.DataFrame'>
RangeIndex: 2142 entries, 0 to 2141
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   folio       2142 non-null   int64
 1   foliohog    2142 non-null   int64
 2   id_calefac  2142 non-null   int64
 3   calef_tipo  2142 non-null   int64
 4   calef_otro  2142 non-null   str  
 5   calef_dia   2142 non-null   int64
 6   calef_hor   2142 non-null   int64
 7   calef_min   2142 non-null   int64
 8   calef_adqu  2142 non-null   int64
 9   calef_anio  2142 non-null   int64
dtypes: int64(9), str(1)
memory usage: 167.5 KB


In [1425]:
c.isnull().sum()
#c.duplicated().sum()

folio         0
foliohog      0
id_calefac    0
calef_tipo    0
calef_otro    0
calef_dia     0
calef_hor     0
calef_min     0
calef_adqu    0
calef_anio    0
dtype: int64

In [1426]:
# filtrar información 

calefactor=c[["folio","id_calefac"]]
print(calefactor)

      folio  id_calefac
0        28           1
1        39           1
2        57           1
3        58           1
4        68           1
...     ...         ...
2137  28808           1
2138  28830           1
2139  28850           1
2140  28870           1
2141  28901           1

[2142 rows x 2 columns]


In [1427]:
print(calefactor[["folio"]].duplicated().sum())

411


In [1428]:
calefactor_2=calefactor.groupby("folio")["id_calefac"].max().reset_index()
print(calefactor_2)

      folio  id_calefac
0        28           1
1        39           1
2        57           1
3        58           1
4        68           1
...     ...         ...
1726  28808           1
1727  28830           1
1728  28850           1
1729  28870           1
1730  28901           1

[1731 rows x 2 columns]


### 4.- DF con información sobre focos

In [1429]:
f.info()

<class 'pandas.DataFrame'>
RangeIndex: 230152 entries, 0 to 230151
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   folio       230152 non-null  int64
 1   foliohog    230152 non-null  int64
 2   id_focos    230152 non-null  int64
 3   foco_num    230152 non-null  int64
 4   foco_hor    230152 non-null  str  
 5   foco_min    230152 non-null  str  
 6   foco_fluor  230152 non-null  str  
 7   foco_led    230152 non-null  str  
 8   foco_incan  230152 non-null  str  
dtypes: int64(4), str(5)
memory usage: 15.8 MB


In [1430]:
#f.duplicated().sum()
f.isnull().sum()

folio         0
foliohog      0
id_focos      0
foco_num      0
foco_hor      0
foco_min      0
foco_fluor    0
foco_led      0
foco_incan    0
dtype: int64

In [1431]:
focos=f[["folio","foco_num"]]
focos.sample(3)

,folio,foco_num
40191,5046,0
140875,17722,1
40947,5142,1


In [1432]:
focos["folio"].duplicated().sum()

np.int64(201383)

In [1433]:
focos_2=focos.groupby("folio")["foco_num"].sum().reset_index()
print(focos_2)

       folio  foco_num
0          1        10
1          2         8
2          3        10
3          4         8
4          5         7
...      ...       ...
28764  28949         5
28765  28950         4
28766  28951         5
28767  28952         4
28768  28953         4

[28769 rows x 2 columns]


In [1434]:
focos_2["folio"].duplicated().sum()

np.int64(0)

### 5.- información sobre región donde se ubica la vivienda

In [1435]:
v.info()

<class 'pandas.DataFrame'>
RangeIndex: 28953 entries, 0 to 28952
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   folio       28953 non-null  int64
 1   tot_hog     28953 non-null  int64
 2   tam_loc     28953 non-null  int64
 3   est_socio   28953 non-null  int64
 4   est_dis     28953 non-null  int64
 5   upm         28953 non-null  int64
 6   factor_sem  28953 non-null  int64
 7   region      28953 non-null  int64
 8   entidad     28953 non-null  int64
dtypes: int64(9)
memory usage: 2.0 MB


In [1436]:
v.isnull().sum()

folio         0
tot_hog       0
tam_loc       0
est_socio     0
est_dis       0
upm           0
factor_sem    0
region        0
entidad       0
dtype: int64

In [1437]:
region=v[["folio","region"]]

In [1438]:
print(region)

       folio  region
0          1       2
1          2       2
2          3       2
3          4       2
4          5       2
...      ...     ...
28948  28949       2
28949  28950       2
28950  28951       2
28951  28952       2
28952  28953       2

[28953 rows x 2 columns]


### 6.- DF con información de uso de refrigerador y consumo de energía

In [1439]:
e.info()

#bastantes columnas y 28,953

<class 'pandas.DataFrame'>
RangeIndex: 28953 entries, 0 to 28952
Columns: 228 entries, folio to mej_habito
dtypes: float64(171), int64(49), str(8)
memory usage: 50.4 MB


In [1440]:
encevi=e[["folio","cons_med1","uso_refri","num_tvs"]]

print(encevi)

       folio  cons_med1  uso_refri  num_tvs
0          1      261.0        1.0      2.0
1          2      150.0        1.0      2.0
2          3      500.0        1.0      2.0
3          4      267.0        1.0      1.0
4          5      420.0        1.0      1.0
...      ...        ...        ...      ...
28948  28949      185.0        1.0      1.0
28949  28950      180.0        1.0      1.0
28950  28951      188.0        1.0      1.0
28951  28952       49.0        1.0      1.0
28952  28953       50.0        1.0      1.0

[28953 rows x 4 columns]


In [1441]:
encevi["uso_refri"] = encevi["uso_refri"].map({
    1: 1,  # sí
    2: 0   # no
})

In [1442]:
encevi["uso_refri"].unique()

array([ 1.,  0., nan])

### Juntar información relevante en un DF

In [1443]:
inf = (
    encevi
    .merge(habitantes, on="folio", how="left")
    .merge(calefactor_2, on="folio", how="left")
    .merge(focos_2, on="folio", how="left")
    .merge(aire_acond, on="folio", how="left")
    .merge(region, on="folio", how="left")
)


In [1444]:
inf.info()

<class 'pandas.DataFrame'>
RangeIndex: 28953 entries, 0 to 28952
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   folio       28953 non-null  int64  
 1   cons_med1   25983 non-null  float64
 2   uso_refri   28769 non-null  float64
 3   num_tvs     26274 non-null  float64
 4   id_pobla    28953 non-null  int64  
 5   id_calefac  1731 non-null   float64
 6   foco_num    28769 non-null  float64
 7   id_aire     5225 non-null   float64
 8   region      28953 non-null  int64  
dtypes: float64(6), int64(3)
memory usage: 2.0 MB


In [1445]:
inf["folio"].duplicated().sum()

np.int64(0)

In [1446]:
inf.sample(3)

,folio,cons_med1,uso_refri,num_tvs,id_pobla,id_calefac,foco_num,id_aire,region
13330,13331,85.0,0.0,2.0,5,NaN,4.0,NaN,2
16620,16621,487.0,1.0,3.0,3,2.0,11.0,2.0,1
24389,24390,51.0,0.0,NaN,2,NaN,3.0,NaN,3


In [1447]:
inf["folio"].nunique()
# 28,953 viviendas (hogares) estrevistadas 

28953

### Modificar nombre columnas

In [1448]:
inf=inf.rename(columns={"folio":"vivienda",
                        "cons_med1":"consumo_electricidad",
                        "uso_refri":"refrigerador",
                       "id_pobla":"ocupantes",
                       "id_calefac": "calefactor",
                       "foco_num": "focos",
                       "id_aire": "aire_acon"}
                       )

inf.sample(4)

,vivienda,consumo_electricidad,refrigerador,num_tvs,ocupantes,calefactor,focos,aire_acon,region
9251,9252,89.0,1.0,3.0,3,NaN,7.0,NaN,2
508,509,467.0,0.0,1.0,4,NaN,5.0,NaN,2
12055,12056,90.0,1.0,NaN,3,NaN,7.0,NaN,2
24053,24054,NaN,1.0,1.0,3,NaN,5.0,NaN,3


In [1449]:
inf.isnull().sum()

vivienda                    0
consumo_electricidad     2970
refrigerador              184
num_tvs                  2679
ocupantes                   0
calefactor              27222
focos                     184
aire_acon               23728
region                      0
dtype: int64

### Tratamiento valores nulos

In [1450]:
inf["consumo_electricidad"]=inf["consumo_electricidad"].fillna(0)
inf["refrigerador"]=inf["refrigerador"].fillna(0)
inf["calefactor"]=inf["calefactor"].fillna(0)
inf["focos"]=inf["focos"].fillna(0)
inf["aire_acon"]=inf["aire_acon"].fillna(0)
inf["num_tvs"]=inf["num_tvs"].fillna(0)

In [1451]:
inf["consumo_electricidad"]=inf["consumo_electricidad"].astype(float)
inf["refrigerador"]=inf["refrigerador"].astype(int)
inf["num_tvs"]=inf["num_tvs"].astype(int)

In [1452]:
inf.isnull().sum()


vivienda                0
consumo_electricidad    0
refrigerador            0
num_tvs                 0
ocupantes               0
calefactor              0
focos                   0
aire_acon               0
region                  0
dtype: int64

In [1453]:
inf.info()

<class 'pandas.DataFrame'>
RangeIndex: 28953 entries, 0 to 28952
Data columns (total 9 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   vivienda              28953 non-null  int64  
 1   consumo_electricidad  28953 non-null  float64
 2   refrigerador          28953 non-null  int64  
 3   num_tvs               28953 non-null  int64  
 4   ocupantes             28953 non-null  int64  
 5   calefactor            28953 non-null  float64
 6   focos                 28953 non-null  float64
 7   aire_acon             28953 non-null  float64
 8   region                28953 non-null  int64  
dtypes: float64(4), int64(5)
memory usage: 2.0 MB


In [1454]:
# filtrar filas que tengan un registro en conusmo de electricidad
inf = inf[inf["consumo_electricidad"] > 0]

In [1455]:
inf = inf[inf["consumo_electricidad"] < inf["consumo_electricidad"].quantile(0.99)]

In [1456]:
dummies = pd.get_dummies(inf["region"], prefix="region", drop_first=True)

inf = pd.concat([inf, dummies], axis=1)
inf = inf.drop(columns=["region"])

In [1457]:
inf.info()

<class 'pandas.DataFrame'>
Index: 24882 entries, 0 to 28952
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   vivienda              24882 non-null  int64  
 1   consumo_electricidad  24882 non-null  float64
 2   refrigerador          24882 non-null  int64  
 3   num_tvs               24882 non-null  int64  
 4   ocupantes             24882 non-null  int64  
 5   calefactor            24882 non-null  float64
 6   focos                 24882 non-null  float64
 7   aire_acon             24882 non-null  float64
 8   region_2              24882 non-null  bool   
 9   region_3              24882 non-null  bool   
dtypes: bool(2), float64(4), int64(4)
memory usage: 1.8 MB


### definir variables:
### dato objetivo a predecir: consumo de energía
### datos matriz: refrigerador, ocupantes, calefactor, focos, aire_acond etc


In [1458]:
X=inf[["refrigerador", "ocupantes", "calefactor", "focos", "aire_acon", "region_2", "region_3","num_tvs"]]
y= inf["consumo_electricidad"]



In [1459]:
# datos entrenamiento

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0)

In [1460]:
modelo = LinearRegression()
modelo.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [1461]:
pred = modelo.predict(X_test)

In [1462]:
# error absoluto 
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print("MAE:", mae)
print("R2:", r2)


MAE: 312.7760009160526
R2: 0.03401431600632132


In [1463]:
coef = pd.DataFrame({
    "variable": X.columns,
    "coeficiente": modelo.coef_
})

print(coef)

       variable  coeficiente
0  refrigerador    31.732373
1     ocupantes    20.502798
2    calefactor    10.599242
3         focos    25.777622
4     aire_acon   281.874577
5      region_2   -76.085179
6      region_3    77.641111
7       num_tvs    39.620697


In [1464]:
error_relativo = mae / y_test.mean()
print(error_relativo)

0.6677502773456161


In [1465]:
inf["consumo_electricidad"].describe()

count    24882.000000
mean       477.280122
std       1584.311226
min          1.000000
25%        160.000000
50%        266.000000
75%        486.000000
max      99899.000000
Name: consumo_electricidad, dtype: float64

In [1466]:
import pandas as pd

importancias = pd.DataFrame({
    "variable": X.columns,
    "importancia": modelo_rf.feature_importances_
}).sort_values(by="importancia", ascending=False)

print(importancias)

       variable  importancia
4     aire_acon     0.486008
7       num_tvs     0.199594
3         focos     0.115143
1     ocupantes     0.073405
2    calefactor     0.056882
5      region_2     0.033636
6      region_3     0.028931
0  refrigerador     0.006401


### Dentro del modelo la variable que más influye en el consumo eléctrico por vivienda es la existencia de aire acondicionado, suena lógico y poco influye si se tiene o no refrigerador.

In [1467]:
print(y_test.mean())
print(mae)
print(mae / y_test.mean())

468.40265220012054
312.7760009160526
0.6677502773456161


### Consideraciones:

1.- refrigerador: indicar si se cuenta con refrigerador, capturar 1=si, 0= no.

2.- ocupantes: capturar número de ocupantes

3.- calefactor: capturar el número de calefactores,

4.- focos: capturar el número de focos,

5.- aire_acond: capturar el númeor de aa

6.- region
    capturar 1 en region_1 (si es región cálida extrema), si no, capturar 0
    capturar 1 en region_2 (si es región templada), si no, capturar 0
    capturar 1 en region_3 (si es región tropical), si no, capturar 0

7.- num_tvs: captura el número de tvs que hay en tu vivienda

In [1468]:
predecir_ejemplo = pd.DataFrame([{
    "refrigerador": 1,
    "ocupantes": 3,
    "calefactor": 0,
    "focos": 6,
    "aire_acon": 0,
    "region_2": 0,
    "region_3": 0,
    "num_tvs": 3
}])

In [1469]:
prediccion = modelo.predict(predecir_ejemplo)
print(prediccion)

[446.32762934]


### CONCLUSIÓN
### El modelo presenta un error promedio de 312 unidades, lo que equivale aproximadamente al 67% del consumo promedio. Este resultado indica que las variables consideradas explican solo una pequeña parte de la variabilidad del consumo eléctrico. El consumo depende de factores adicionales no incluidos en el modelo, como características socioeconómicas, estructurales del hogar o patrones de uso de la energía.